In [56]:
import boto3
import sagemaker
import pandas as pd
import awswrangler as wr
import time
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket = sess.default_bucket()

# setup the IAM, so it gives featurestore access to the s3 bucket
role = get_execution_role()
region = sess.boto_region_name

s3 = boto3.client("s3")
boto_session = boto3.Session(region_name=region)

sm = boto_session.client(service_name="sagemaker",region_name = region)
featurestore_runtime = boto_session.client(service_name = "sagemaker-featurestore-runtime", region_name = region)

feature_store_session = Session(
    boto_session= boto_session,
    sagemaker_client= sm,
    sagemaker_featurestore_runtime_client= featurestore_runtime
)

## S3 bucket for the offline store

In [57]:
default_s3_bucket  = feature_store_session.default_bucket()

In [58]:
default_s3_bucket

'sagemaker-us-east-1-103012382341'

## Inspect Data

In [59]:

# Load training data
train_df = wr.s3.read_parquet(f"s3://{bucket}/airline-delay/training/data.parquet")
print(f"Train shape: {train_df.shape}")
print(train_df.dtypes)

Train shape: (650420, 20)
year                    Int64
month                   Int64
dayofmonth              Int64
dayofweek               Int64
reporting_airline       Int64
origin                 string
dest                   string
originstate             Int64
deststate               Int64
crselapsedtime        float64
distance              float64
arrdel15                Int64
dep_hour                Int64
is_weekend              Int64
route                  string
carrier_delay_rate    float64
origin_delay_rate     float64
dest_delay_rate       float64
route_delay_rate      float64
hour_delay_rate       float64
dtype: object


In [60]:
train_df.head()

,year,month,dayofmonth,dayofweek,reporting_airline,origin,dest,originstate,deststate,crselapsedtime,distance,arrdel15,dep_hour,is_weekend,route,carrier_delay_rate,origin_delay_rate,dest_delay_rate,route_delay_rate,hour_delay_rate
0,2024,1,1,1,4,ATL,SAN,8,4,300.0,1892.0,0,18,0,ATL_SAN,0.172640,0.187492,0.214648,0.101695,0.283663
1,2024,1,1,1,11,IAH,ASE,43,5,180.0,913.0,0,16,0,IAH_ASE,0.233288,0.240598,0.340820,0.365385,0.268016
2,2024,1,1,1,8,FSM,DFW,2,43,81.0,227.0,1,6,0,FSM_DFW,0.247266,0.339623,0.279543,0.339623,0.136619
3,2024,1,1,1,11,SLC,SFO,44,4,158.0,599.0,0,8,0,SLC_SFO,0.233288,0.224608,0.291255,0.323782,0.178455
4,2024,1,1,1,11,ASE,SFO,5,4,179.0,848.0,0,10,0,ASE_SFO,0.233288,0.342155,0.291255,0.376812,0.209014


## Injest Data into FeatureStore

 Feature Store needs two extra columns — a unique record ID and an event time

In [62]:

# Add required Feature Store columns
train_df = train_df.copy()
train_df["flight_id"] = [str(i) for i in range(len(train_df))]
train_df["event_time"] = "2024-01-01T00:00:00Z"

# Fix types — Feature Store requires no Int64/string ambiguity
for col in ["year","month","dayofmonth","dayofweek","reporting_airline",
          "originstate","deststate","dep_hour","is_weekend","arrdel15"]:
  train_df[col] = train_df[col].astype(int)

for col in ["crselapsedtime","distance","carrier_delay_rate","origin_delay_rate",
          "dest_delay_rate","route_delay_rate","hour_delay_rate"]:
  train_df[col] = train_df[col].astype(float)

for col in ["origin","dest","route"]:
  train_df[col] = train_df[col].astype(str)

print(f"Shape with new cols: {train_df.shape}")
print(train_df[["flight_id","event_time"]].head(2))

Shape with new cols: (650420, 22)
  flight_id            event_time
0         0  2024-01-01T00:00:00Z
1         1  2024-01-01T00:00:00Z


In [63]:
record_identifier_feature_name = "flight_id"
event_time_feature_name = "event_time"

In [64]:
%store -r s3_aerodelay

Map Pandas Column Data Types to explicit V3 FeatureDefinition validation shapes

In [66]:
from sagemaker.core.shapes.shapes import FeatureDefinition, OfflineStoreConfig, S3StorageConfig

feature_definitions = fs.load_feature_definitions_from_dataframe(data_frame=train_df)
feature_group_name= "aerodelay-flight"

feature_group = fs.FeatureGroup.create(
  feature_group_name=feature_group_name,
  record_identifier_feature_name= record_identifier_feature_name,
  event_time_feature_name= event_time_feature_name,
  feature_definitions=feature_definitions,
  offline_store_config=OfflineStoreConfig(
      s3_storage_config=S3StorageConfig(
          s3_uri=f"{s3_aerodelay}/feature-store/"
      )
  ),
  role_arn=role,
)

print("Waiting for Feature Group...")
while True:
  status = sm.describe_feature_group(
      FeatureGroupName=feature_group_name
  )["FeatureGroupStatus"]
  print(f"  Status: {status}")
  if status == "Created":
      break
  if status == "CreateFailed":
      desc = sm_client.describe_feature_group(FeatureGroupName=feature_group_name)
      raise RuntimeError(desc.get("FailureReason", "Unknown"))
  time.sleep(10)

print("Feature Group ready.")

[05/31/26 23:53:22] INFO     Creating feature_group resource.                                    ]8;id=5707807;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5707808;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11591\11591]8;;\

Waiting for Feature Group...
  Status: Creating


  Status: Creating


  Status: Creating


  Status: Created
Feature Group ready.


In [67]:
import sagemaker.mlops.feature_store as fs

fs.ingest_dataframe(
  feature_group_name=feature_group_name,
  data_frame=train_df,
  max_workers=4,
)
print(f"Ingested {len(train_df):,} records into {feature_group_name}")

[05/31/26 23:53:58] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5707813;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5707814;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5707819;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5707820;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Ingested 650,420 records into aerodelay-flight


## Verify Data in Offline Store

In [68]:
desc = sm.describe_feature_group(FeatureGroupName=feature_group_name)

print(f"Feature Group Status: {desc['FeatureGroupStatus']}")
print(f"Offline Store Status: {desc.get('OfflineStoreStatus', {}).get('Status', 'N/A')}")
print(f"Offline S3 URI: {desc['OfflineStoreConfig']['S3StorageConfig']['S3Uri']}")

Feature Group Status: Created
Offline Store Status: Active
Offline S3 URI: s3://sagemaker-us-east-1-103012382341/airline-delay/feature-store/


In [69]:
%store -r aerodelay_s3_staging_dir

In [70]:
import awswrangler as wr

ft_table = desc["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"]
ft_database = desc["OfflineStoreConfig"]["DataCatalogConfig"]["Database"]

print(f"Querying: {ft_database}.{ft_table}")

query = f"""
SELECT COUNT(*) as total_records,
     AVG(CAST(arrdel15 AS double)) as delay_rate,
     MIN(month) as min_month,
     MAX(month) as max_month
FROM "{ft_database}"."{ft_table}"
"""

df_check = wr.athena.read_sql_query(
  sql=query,
  database=ft_database,
  s3_output=aerodelay_s3_staging_dir,
)
print(df_check)


Querying: sagemaker_featurestore.aerodelay_flight_1780271602


[06/01/26 00:32:45] INFO     Created CTAS table                                                       ]8;id=5707825;file:///opt/conda/lib/python3.12/site-packages/awswrangler/athena/_utils.py\_utils.py]8;;\:]8;id=5707826;file:///opt/conda/lib/python3.12/site-packages/awswrangler/athena/_utils.py#891\891]8;;\
                             "sagemaker_featurestore"."temp_table_16fa9b09ecfe4636a0144b2b2d7958cd"                

   total_records  delay_rate  min_month  max_month
0         650420    0.225394          1          2


In [54]:
%store ft_database ft_table

Stored 'ft_database' (str)
Stored 'ft_table' (str)


In [72]:
%store -r aerodelay_s3_staging_dir

In [73]:
%store

Stored variables and their in-db values:
aerodelay_db                         -> 'aerodelay'
aerodelay_s3_staging_dir             -> 's3://sagemaker-us-east-1-103012382341/athena/stag
aerodelay_table                      -> 'flights'
ft_database                          -> 'sagemaker_featurestore'
ft_table                             -> 'aerodelay_flights_1780269129'
s3_aerodelay                         -> 's3://sagemaker-us-east-1-103012382341/airline-del


In [74]:
query = f"""
SELECT * FROM "{ft_database}"."{ft_table}"
"""

training_data = wr.athena.read_sql_query(
  sql=query,
  database=ft_database,
  s3_output=aerodelay_s3_staging_dir,
)

# Drop Feature Store metadata columns
training_data = training_data.drop(
  columns=["write_time", "api_invocation_time", "is_deleted", "event_time", "flight_id"],
  errors="ignore"
)

print(f"Training dataset shape: {training_data.shape}")
print(f"Columns: {list(training_data.columns)}")

# Save to S3 for teammates
wr.s3.to_parquet(
  df=training_data,
  path=f"{s3_aerodelay}/training_from_feature_store/data.parquet",
)
print("Saved to S3 for teammates.")

[06/01/26 00:48:41] INFO     Created CTAS table                                                       ]8;id=5707831;file:///opt/conda/lib/python3.12/site-packages/awswrangler/athena/_utils.py\_utils.py]8;;\:]8;id=5707832;file:///opt/conda/lib/python3.12/site-packages/awswrangler/athena/_utils.py#891\891]8;;\
                             "sagemaker_featurestore"."temp_table_c8eb404871374a95b46a6e0587c1b447"                

Training dataset shape: (650420, 20)
Columns: ['year', 'month', 'dayofmonth', 'dayofweek', 'reporting_airline', 'origin', 'dest', 'originstate', 'deststate', 'crselapsedtime', 'distance', 'arrdel15', 'dep_hour', 'is_weekend', 'route', 'carrier_delay_rate', 'origin_delay_rate', 'dest_delay_rate', 'route_delay_rate', 'hour_delay_rate']


Saved to S3 for teammates.
